# Qubitized Hubbard Select

Subroutines for the SELECT operation for the Hubbard model Hamiltonian.

This module follows section V. of Encoding Electronic Spectra in Quantum Circuits with Linear T
Complexity. Babbush et al. 2018. [arxiv:1805.03662](https://arxiv.org/abs/1805.03662).


The goal is to construct a SELECT operator optimized for the following
Hamiltonian:

$$
\def\Zvec{\overrightarrow{Z}}
\def\hop#1{#1_{p,\sigma} \Zvec #1_{q,\sigma}}
H = -\frac{t}{2} \sum_{\langle p,q \rangle, \sigma} (\hop{X} + \hop{Y})
  + \frac{u}{8} \sum_{p,\alpha\ne\beta} Z_{p,\alpha}Z_{p,\beta}
  - \frac{u}{4} \sum_{p,\sigma} Z_{p,\sigma} + \frac{uN}{4}\mathbb{1}
$$

With these operators, our selection register has indices
for $p$, $\alpha$, $q$, and $\beta$ as well as two indicator bits $U$ and $V$. There are four cases
considered in both the PREPARE and SELECT operations corresponding to the terms in the Hamiltonian:

 - $U=1$, single-body Z
 - $V=1$, spin-spin ZZ term
 - $p<q$, XZX term
 - $p>q$, YZY term.
Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.


In [ ]:
from dataclasses import asdict

from IPython.display import SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.hubbard_qubitization_resources import (
    notebook_models, notebook_signature,
)
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg
from flagquantum.qec.resources.estimation import estimate_ftqc_resources


def show_resource_graph(model):
    dot = export_modeled_call_graph_dot(model, max_depth=1)
    display(SVG(render_call_graph_svg(dot)))
    report = estimate_ftqc_resources(model)
    assert report.emitted is None
    print("Logical gate estimate", asdict(report.logical.gates))
    print("Peak logical qubits", report.logical.peak_logical_qubits)


## `SelectHubbard`

SELECT acts on the system indexed by the two coordinate and spin addresses.
It composes two Majorana operations, swaps the selection coordinates where
required, corrects the Pauli product phase, and applies the spin-up Z term.
The target order is spin, y, then x with x varying fastest.

In [ ]:
from flagquantum.algorithms.primitives.hubbard_select import SelectHubbard

### Example Instances

In [ ]:
select = SelectHubbard(4)
select_signature = notebook_signature(4)
_, select_model = notebook_models()[0]
assert select.source_resource_model() == select_model

In [ ]:
controlled = SelectHubbard(4, controlled=True)
controlled_signature = notebook_signature(4, controlled=True)
assert controlled.source_resource_model().qubits == controlled_signature.n_qubits()

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("4×4 SELECT", select_signature, {}),
    ("4×4 controlled SELECT", controlled_signature, {}),
])))

### Call Graph

In [ ]:
show_resource_graph(select_model)

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
from IPython.display import Image

from examples.algorithms.hubbard_qubitization_resources import notebook_controlled_select_score
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score

In [ ]:
large_select = SelectHubbard(11, controlled=True)
assert large_select.source_resource_model() == notebook_models()[-1][1]
score = notebook_controlled_select_score()
figure, _ = draw_register_score(register_score_data(score), unit_to_inches=1.75, max_width=24, max_height=9)
buffer = BytesIO()
figure.savefig(buffer, format="png", bbox_inches="tight")
plt.close(figure)
display(Image(data=buffer.getvalue()))

## Hubbard Majorana operator

The symbolic signature describes a controlled X consumer. The small
concrete example uses Y and includes the Jordan–Wigner Z prefix.
The declared resource formula counts the unary traversal and erasure.
The controlled Pauli leaves appear in the decomposition separately.

In [ ]:
from sympy import Symbol
from examples.algorithms.hubbard_qubitization_resources import notebook_consumer_signature
from flagquantum.algorithms.primitives.hubbard_qubitization_resource_model import (
    source_hubbard_majorana_formula,
)
from flagquantum.qec.resources.estimation import estimate_symbolic_ftqc_resources

### Example Instances

In [ ]:
n = Symbol("n")
majorana = notebook_consumer_signature(n, majorana=True, controlled=True)
majorana_model = source_hubbard_majorana_formula(n, controlled=True)

In [ ]:
small_majorana = notebook_consumer_signature(3, majorana=True)
small_majorana_model = source_hubbard_majorana_formula(3)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Controlled Majorana X", majorana, {}),
    ("3×3 Majorana Y", small_majorana, {}),
])))
print("Symbolic logical resources", asdict(estimate_symbolic_ftqc_resources(majorana_model).logical))
print("Concrete logical resources", asdict(small_majorana_model))

## Hubbard spin-up Z

The spin-up consumer selects half of the system bank and applies Z at
the requested coordinate when the interaction selector is active.

In [ ]:
from flagquantum.algorithms.primitives.hubbard_qubitization_resource_model import (
    source_hubbard_spin_up_formula,
)

### Example Instances

In [ ]:
spin_up = notebook_consumer_signature(n, majorana=False, controlled=True)
spin_up_model = source_hubbard_spin_up_formula(n, controlled=True)

In [ ]:
small_spin_up = notebook_consumer_signature(3, majorana=False)
small_spin_up_model = source_hubbard_spin_up_formula(3)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Controlled spin-up Z", spin_up, {}),
    ("3×3 spin-up Z", small_spin_up, {}),
])))
print("Symbolic logical resources", asdict(estimate_symbolic_ftqc_resources(spin_up_model).logical))
print("Concrete logical resources", asdict(small_spin_up_model))